# Importing Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# load the dataset

In [ ]:
df = pd.read_csv("data/Churn Sampled.csv")

df = df[["Geography", "Gender","Card Type","Age","CreditScore","Tenure","Point Earned","Balance","EstimatedSalary", 
    "NumOfProducts","IsActiveMember","HasCrCard","Satisfaction Score","Exited"]]

# df = df[["Geography", "Gender","Card Type","Age","Complain","CreditScore","Tenure","Point Earned","Balance","EstimatedSalary", 
#     "NumOfProducts","IsActiveMember","HasCrCard","Satisfaction Score","Exited"]]

df.sample(10)

# Split dependent & Independant Data

In [ ]:
X = df.drop("Exited", axis = 1)
y = df["Exited"]

# Train, Validation and Test Split

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(X, y, train_size = 0.8, stratify = y, random_state = 42)

In [ ]:
xtrain.shape,xtest.shape, ytrain.shape, ytest.shape

In [ ]:
xtrain, xval, ytrain, yval = train_test_split(xtrain, ytrain, train_size = 0.85, 
                                            stratify = ytrain, random_state = 42)

In [ ]:
xtrain.shape, xval.shape, ytrain.shape, yval.shape 

In [ ]:
print(xtrain["Geography"].unique(),xval["Geography"].unique(),xtest["Geography"].unique())
print(xtrain["Gender"].unique(),xval["Gender"].unique(),xtest["Gender"].unique())
print(xtrain["Card Type"].unique(),xval["Card Type"].unique(),xtest["Card Type"].unique())

# Preprocessing

## Encodings

In [ ]:
for col in df.select_dtypes(include= "object").columns:
    print(col, ":",df[col].unique())

In [ ]:
# le_geography = LabelEncoder()
# le_gender = LabelEncoder()
# oe_cardtype = OrdinalEncoder(categories = [["SILVER", "GOLD", "DIAMOND", "PLATINUM"]])

# # Geography
# xtrain["Geography"] = le_geography.fit_transform(xtrain["Geography"])
# xval["Geography"] = le_geography.transform(xval["Geography"])
# xtest["Geography"] = le_geography.transform(xtest["Geography"])

# # Gender
# xtrain["Gender"] = le_gender.fit_transform(xtrain["Gender"])
# xval["Gender"] = le_gender.transform(xval["Gender"])
# xtest["Gender"] = le_gender.transform(xtest["Gender"])

# # Card Type
# xtrain["Card Type"] = oe_cardtype.fit_transform(xtrain[["Card Type"]])
# xval["Card Type"] = oe_cardtype.transform(xval[["Card Type"]])
# xtest["Card Type"] = oe_cardtype.transform(xtest[["Card Type"]])

## Scaling

In [ ]:
# # Age, CreditScore, Point Earned, Balance, EstimatedSalary
# mms = MinMaxScaler()
# xtrain[["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]] \
#             = mms.fit_transform(xtrain[["Age","CreditScore", "Point Earned", "Balance", "EstimatedSalary"]])
# xval[["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]] \
#             = mms.transform(xval[["Age","CreditScore", "Point Earned", "Balance", "EstimatedSalary"]])
# xtest[["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]] \
#             = mms.transform(xtest[["Age","CreditScore", "Point Earned", "Balance", "EstimatedSalary"]])

In [ ]:
# gbc = GradientBoostingClassifier(**{'subsample': 0.8, 'n_estimators': 150, 'min_samples_split': 4, 'max_depth': 3, 'learning_rate': 0.05})
# xgb = XGBClassifier(**{'subsample': 1.0, 'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.2, 'gamma': 0.2, 'colsample_bytree': 0.5})
# rfc = RandomForestClassifier(**{'n_estimators': 200, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None})

# gbc.fit(xtrain, ytrain)
# xgb.fit(xtrain, ytrain)
# rfc.fit(xtrain, ytrain)

# Model Pipeline

In [ ]:
# Feature Seperation
le_features = ["Geography", "Gender"]
oe_features = ["Card Type"]
scale_features = ["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]
pass_features = ["Tenure", "NumOfProducts", "IsActiveMember", "HasCrCard", "Satisfaction Score"]

In [ ]:
# Preprocessing
preprocessor = ColumnTransformer([
    ("LE Geo", OrdinalEncoder(), le_features), 
    ("OE CardType", OrdinalEncoder(categories=[["SILVER", "GOLD", "DIAMOND", "PLATINUM"]]), oe_features),
    ("MMScaler", MinMaxScaler(), scale_features)
], remainder="passthrough")
preprocessor

In [ ]:
# Gradient Boosting Pipeline
gbc_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        subsample=0.8, n_estimators=150, min_samples_split=4,
        max_depth=3, learning_rate=0.05, random_state=42))
])
gbc_pipeline

In [ ]:
# XGBoost Pipeline
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        subsample=1.0, n_estimators=50, max_depth=3,
        learning_rate=0.2, gamma=0.2, colsample_bytree=0.5, eval_metric='logloss', random_state=42))
])
xgb_pipeline

In [ ]:
# Random Forest Pipeline
rfc_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, min_samples_split=6, min_samples_leaf=1,
        max_features='sqrt', max_depth=None, random_state=42))
])
rfc_pipeline

In [ ]:
gbc_pipeline.fit(xtrain, ytrain)

In [ ]:
xgb_pipeline.fit(xtrain, ytrain)

In [ ]:
rfc_pipeline.fit(xtrain, ytrain)

In [ ]:
confusion_matrix(yval, gbc_pipeline.predict(xval)), confusion_matrix(ytest, gbc_pipeline.predict(xtest))

In [ ]:
confusion_matrix(yval, xgb_pipeline.predict(xval)), confusion_matrix(ytest, xgb_pipeline.predict(xtest))

In [ ]:
confusion_matrix(yval, rfc_pipeline.predict(xval)), confusion_matrix(ytest, rfc_pipeline.predict(xtest))

# Saving The Models

In [ ]:
import pickle

with open("models/GBC_Pipeline.pkl", "wb") as f:
    pickle.dump(obj = gbc_pipeline, file = f)

with open("models/XGB_Pipeline.pkl", "wb") as f:
    pickle.dump(obj = xgb_pipeline, file = f)

with open("models/RFC_Pipeline.pkl", "wb") as f:
    pickle.dump(obj = rfc_pipeline, file = f)